# 01 · Data Ingestion — EIA Form 930

Downloads all EIA Form 930 hourly grid balance CSV files (2019–2024) from the
[EIA Grid Monitor bulk-files page](https://www.eia.gov/electricity/gridmonitor/about).

**Outputs**: `data/raw/EIA930_BALANCE_<YEAR>_<HALF>.csv` — 12 semi-annual files, ~200–400 MB each.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from src.ingest import build_manifest, run_ingest, RAW_DIR

print(f'Raw data directory: {RAW_DIR.resolve()}')
print(f'Manifest size: {len(build_manifest())} files')

In [ ]:
# Show the full download manifest before running
manifest = build_manifest()
for url, dest in manifest:
    status = '✓ exists' if dest.exists() else '  missing'
    print(f'{status}  {dest.name}')

In [ ]:
# Download all files (skips existing)
# Estimated total: ~3-5 GB, ~20-40 min depending on connection.
result = run_ingest(force=False)
print(result)

In [ ]:
# Verify downloads and inspect one file
import pandas as pd

csv_files = sorted(RAW_DIR.glob('EIA930_BALANCE_*.csv'))
print(f'Files in data/raw/: {len(csv_files)}')

total_bytes = sum(f.stat().st_size for f in csv_files)
print(f'Total size: {total_bytes / 1e9:.2f} GB')

for f in csv_files:
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:45s}  {size_mb:7.1f} MB')

In [ ]:
# Peek at column schema from the first available file
if csv_files:
    sample = pd.read_csv(csv_files[0], nrows=5)
    print('Columns:', sample.columns.tolist())
    sample

In [ ]:
# Count unique balancing authorities across all years
if csv_files:
    ba_sets = []
    for f in csv_files:
        chunk = pd.read_csv(f, usecols=lambda c: 'respondent' in c.lower() and 'name' not in c.lower(), nrows=None)
        ba_sets.append(set(chunk.iloc[:, 0].dropna().unique()))
    all_bas = set().union(*ba_sets)
    print(f'Unique balancing authorities: {len(all_bas)}')
    print(sorted(all_bas))